# N01: metadata baseline

Can a visitor find a painting by describing what is **in** it ("a stormy sea with big waves")? The dataset only has
genre, style and artist (and the artist is unknown for 66% of the dev paintings), so the baseline searches that.

The ladder, one notebook per step, each compared with the previous ones on the same queries:

| Rung | Searches | Notebook |
|---|---|---|
| R0 random | nothing (the floor) | N01 |
| R1 metadata | genre + style + artist, MiniLM | N01 |
| R2 description | a VLM description of the image, MiniLM | N02 |
| R3 description + metadata | both, MiniLM | N02 |
| R4 SigLIP | the image itself, SigLIP | N03 |
| R5 hybrid | R3 + R4 | N04 |

How we measure (ADR 0007): 40 visual queries written before any result existed, the top 5 of each rung graded blind
(0 no, 1 partial, 2 clear) by an LLM judge validated against human grades (N04). Main metric **nDCG@5** (are the 5 paintings the kiosk shows good, best first?),
plus **Success@5** (is at least one a clear match?). Everything here is on **dev** (Museum A validation, 480
paintings); Museum B is scored once, in N05.

In [ ]:

from deep_learning_project_waerniers_lamy.nlp import utils as u

## 1. Corpus and queries

In [ ]:
paintings = u.load_corpus("dev")
queries = u.QUERIES["text"].tolist()

print(len(paintings), "paintings |", (paintings["artist"] == "Unknown Artist").mean().round(2), "with unknown artist")
print(u.QUERIES["category"].value_counts().to_dict())
print("searchable text of the baseline, e.g.:", u.metadata_text(paintings.iloc[0]))
u.QUERIES.head()

## 2. Rank the paintings

Each rung gives a similarity between every query and every painting; the 5 highest are what the kiosk would show.

In [ ]:
scores = {
    "R0 random": u.rung_random(paintings, queries),
    "R1 metadata": u.rung_metadata(paintings, queries),
}
# What the baseline shows for one query: it can only match words of the metadata.
top = u.top_k_ids(scores["R1 metadata"], paintings)[12]
print(queries[12])
paintings.set_index("id").loc[top, ["genre", "style", "artist"]]

## 3. Grade the new top-5 paintings

In [ ]:
u.judge_with_llm("dev", scores, paintings)

### Judging

Every painting in the new top 5 of a rung is graded by an LLM judge (`u.JUDGE_MODEL`, another model family than the
description generator). It sees **only the query and the image**, answers two yes/no questions (is the main subject
there? does it match the whole query?) and the grade follows by rule: 0, 1 or 2. Grades are saved in
`results/nlp/qrels_dev.json`; already graded pairs are never sent again. The judge is validated against 150 of your own
grades in N04 before any result is frozen.

## 4. Results

A query with nothing relevant judged has no defined nDCG and is dropped (the count is shown). `judged@5` should be 1.0:
every painting shown was judged.

In [ ]:
per_query = u.per_query_scores("dev", scores, paintings)
table = u.summary_table(per_query)
print("queries kept:", per_query["query_id"].nunique(), "of", len(queries))
u.plot_ladder(table, "dev: baseline")
table

What to look for: R1 should beat R0 on queries whose words appear in the metadata (e.g. "a portrait of a child" ->
genre portrait) and be close to random on the rest (colours, weather, objects).

In [ ]:
per_query.groupby(["rung", "category"])["ndcg@5"].mean().unstack().round(3)